# Урок 6. Feature Scaling — масштабирование признаков
**Библиотеки:** numpy, pandas, sklearn, matplotlib

**Цель:** понять, зачем приводить признаки к одному масштабу и как это делать правильно.

## Простыми словами
Площадь: 30–150. Комнаты: 1–5. Признаки в РАЗНЫХ масштабах. Для gradient descent это как идти
по ущелью: по одной оси шаги огромные, по другой крошечные → учится медленно и криво.

**Z-score нормализация (StandardScaler):** x_new = (x − среднее) / std.
После: у каждого признака среднее 0, разброс 1 — все «на равных».

**Золотое правило:** scaler.fit() ТОЛЬКО на train. Иначе информация из test протечёт
в обучение (data leakage) и оценка будет нечестной.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
df = pd.DataFrame({
    "площадь": rng.uniform(30, 150, 200),
    "комнаты": rng.integers(1, 5, 200).astype(float),
})
df["цена"] = 2.5*df["площадь"] + 15*df["комнаты"] + rng.normal(0, 10, 200)

X = df[["площадь", "комнаты"]]
y = df["цена"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)  # fit: запомнил mean и std train; transform: применил
X_test_s  = scaler.transform(X_test)       # ТОЛЬКО transform — с параметрами train!

print("До:  mean =", X_train.mean().round(1).values, " std =", X_train.std().round(1).values)
print("После: mean =", X_train_s.mean(axis=0).round(2), " std =", X_train_s.std(axis=0).round(2))

In [ ]:
# Влияние масштабирования на gradient descent (свой GD из урока 4, 2 признака)
def gd(X, y, alpha, iters=200):
    w = np.zeros(X.shape[1]); b = 0.0
    hist = []
    for _ in range(iters):
        err = X @ w + b - y
        w -= alpha * (X.T @ err) / len(y)
        b -= alpha * err.mean()
        hist.append((err**2).mean()/2)
    return hist

import matplotlib.pyplot as plt
h_raw = gd(X_train.values, y_train.values, alpha=0.00005)  # без масштаба alpha приходится брать крошечный
h_sc  = gd(X_train_s,      y_train.values, alpha=0.1)
plt.plot(h_raw, label='без scaling (alpha=0.00005)')
plt.plot(h_sc,  label='со scaling (alpha=0.1)')
plt.legend(); plt.grid(True); plt.title('Со scaling сходится в разы быстрее'); plt.show()

## Что произошло
Без масштабирования пришлось брать микроскопический alpha (иначе взрыв), и обучение ползёт.
После StandardScaler оба признака сопоставимы → большой alpha безопасен → cost рушится вниз быстро.

Когда scaling нужен: gradient descent, logistic regression, KMeans, KNN, нейросети.
Когда не нужен: деревья и random forest (им всё равно на масштаб).

## Типичные ошибки
1. fit_transform на ВСЕХ данных до split — data leakage.
2. Масштабировать y — обычно не нужно для базовых задач.
3. Забыть transform для test/новых данных — модель получит «сырые» числа и выдаст чушь.

## Конспект 📓
Z-score: (x−mean)/std → mean 0, std 1. scaler.fit_transform(train), scaler.transform(test).
Нужен: GD, лог.регрессия, KMeans, NN. Не нужен: деревья. fit только на train!

---
## Мини-задание 💪
Примени MinMaxScaler вместо StandardScaler. В какой диапазон попали признаки? Выведи min и max после трансформации.

## 5 проверочных вопросов ❓
1. Зачем нужно масштабирование признаков?
2. Формула z-score и что получается после неё?
3. Почему scaler.fit только на train? Как называется эта ошибка?
4. Каким моделям масштабирование не нужно?
5. Что сделать с новой квартирой перед predict, если модель училась на масштабированных данных?

## Домашнее задание 🏠
Задачи 22–23, 42 из practice_tasks.md.

> Не понял что-то? Открой prompts_for_future.md — промпты 1–6 помогут разобрать любую тему с AI.
